# Exercise 1: Hyperparameter tuning

This notebook builds a Fashion-MNIST classifier, establishes a baseline, runs two focused hyperparameter experiments, and analyses the saved TensorBoard logs.

In [2]:
import os
import tomllib

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.optim as optim
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer import ReportTypes, Trainer, TrainerSettings, metrics
from mltrainer.preprocessors import BasePreprocessor
from tensorboard.backend.event_processing import event_accumulator
from torch import nn

### Data

Fashion-MNIST is loaded through `mads_datasets`. A shared preprocessor and batch size are used for all experiments.

In [3]:
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()

2026-03-21 11:31:13.544 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/fblaak/.cache/mads_datasets/fashionmnist
2026-03-21 11:31:13.546 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/fblaak/.cache/mads_datasets/fashionmnist/fashionmnist.pt


### Metric

Accuracy measures classification performance on the validation data.

In [4]:
accuracy = metrics.Accuracy()

## Baseline experiment

The baseline uses batches of 64 images and trains on 100 batches per epoch. Validation also uses 100 batches. TensorBoard stores the metrics, while TOML files store the model and training configuration.

In [5]:
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=100,
    valid_steps=100,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

### Model

The network contains three hidden linear layers with ReLU activations. Their widths are public attributes, so `Trainer` stores them automatically in each run's `model.toml`.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(
        self,
        num_classes: int,
        units1: int,
        units2: int,
        units3: int,
        dropout_rate: float = 0.0,
        normalization: str = "none",
    ) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3
        # expose hyperparameters so Trainer/TOML can log them
        self.dropout_rate = float(dropout_rate)
        self.normalization = str(normalization)

        def block(in_features, out_features):
            layers = [nn.Linear(in_features, out_features)]
            if self.normalization == "batch":
                layers.append(nn.BatchNorm1d(out_features))
            elif self.normalization == "layer":
                layers.append(nn.LayerNorm(out_features))
            layers.append(nn.ReLU())
            if self.dropout_rate and self.dropout_rate > 0.0:
                layers.append(nn.Dropout(self.dropout_rate))
            return layers

        layers = []
        layers.extend(block(28 * 28, self.units1))
        layers.extend(block(self.units1, self.units2))
        layers.extend(block(self.units2, self.units3))
        # final classifier (no activation)
        layers.append(nn.Linear(self.units3, self.num_classes))
        self.model = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        logits = self.model(x)
        return logits

model = NeuralNetwork(num_classes=10, units1=256, units2=256, units3=128)

### Train the baseline

`Trainer` writes TensorBoard events and TOML configuration files automatically because both report types are enabled. No separate save step is needed.

In [9]:
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau
)
trainer.loop()

2026-03-21 11:32:03.864 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260321-113203
2026-03-21 11:32:05.860 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 100/100 [00:00<00:00, 164.31it/s]
2026-03-21 11:32:06.829 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.0043 test 0.6966 metric ['0.7456']
100%|██████████| 100/100 [00:00<00:00, 175.98it/s]
2026-03-21 11:32:07.633 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6107 test 0.5540 metric ['0.8063']
100%|██████████| 100/100 [00:00<00:00, 121.94it/s]
2026-03-21 11:32:08.892 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.5713 test 0.5248 metric ['0.8108']
100%|██████████| 3/3 [00:02<00:00,  1.03it/s]


## Experiment 1: Network architecture

This grid search compares hidden-layer widths. All other training settings remain fixed so the architectures are compared fairly.

The grid contains $3^3 = 27$ architectures. This exhaustive approach is simple but becomes expensive quickly as more hyperparameters are added.

In [ ]:
units = [256, 128, 64]

architecture_settings = TrainerSettings(
    epochs=10,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

for unit1 in units:
    for unit2 in units:
        for unit3 in units:
            model = NeuralNetwork(
                num_classes=10,
                units1=unit1,
                units2=unit2,
                units3=unit3,
            )
            model.experiment = "architecture_grid"
            model.optimizer = "Adam"

            trainer = Trainer(
                model=model,
                settings=architecture_settings,
                loss_fn=loss_fn,
                optimizer=optim.Adam,
                traindataloader=trainstreamer,
                validdataloader=validstreamer,
                scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            )
            trainer.loop()

Each run receives its own directory containing TensorBoard events, `model.toml`, and `settings.toml`. The added `experiment` attribute makes it possible to separate architecture runs from optimizer runs during analysis.

## Experiment 2: Optimizer and learning rate

This experiment keeps the architecture fixed and compares three optimizers at three learning rates. This isolates the interaction between optimizer choice and learning rate.

To inspect training curves interactively, run `tensorboard --logdir=modellogs` from this directory.

In [ ]:
optimizers = {
    "Adam": optim.Adam,
    "SGD": optim.SGD,
    "RMSprop": optim.RMSprop,
}
learning_rates = [1e-2, 1e-3, 1e-4]

for optimizer_name, optimizer_class in optimizers.items():
    for learning_rate in learning_rates:
        optimizer_settings = TrainerSettings(
            epochs=5,
            metrics=[accuracy],
            logdir="modellogs",
            train_steps=100,
            valid_steps=100,
            optimizer_kwargs={"lr": learning_rate},
            reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
        )

        model = NeuralNetwork(num_classes=10, units1=128, units2=64, units3=32)
        model.experiment = "optimizer_grid"
        model.optimizer = optimizer_name

        trainer = Trainer(
            model=model,
            settings=optimizer_settings,
            loss_fn=loss_fn,
            optimizer=optimizer_class,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        )
        trainer.loop()

## Analyse experiment logs

The code below reads all existing log directories, including older runs that do not yet contain an explicit experiment label. Results are split into architecture and optimizer experiments before comparison.

In [ ]:
def get_last_scalar(run_path, tag):
    try:
        accumulator = event_accumulator.EventAccumulator(run_path)
        accumulator.Reload()
        if tag in accumulator.scalars.Keys():
            values = accumulator.scalars.Items(tag)
            if values:
                return values[-1].value
    except Exception as error:
        print(f"Error reading {tag} from {run_path}: {error}")
    return None


logdir = "modellogs"
rows = []

for run in sorted(os.listdir(logdir)):
    run_path = os.path.join(logdir, run)
    if not os.path.isdir(run_path):
        continue

    model_file = os.path.join(run_path, "model.toml")
    settings_file = os.path.join(run_path, "settings.toml")
    if not (os.path.exists(model_file) and os.path.exists(settings_file)):
        continue

    with open(model_file, "rb") as file:
        model_config = tomllib.load(file)["model"]
    with open(settings_file, "rb") as file:
        training_config = tomllib.load(file)["model"]

    optimizer_name = model_config.get("optimizer")
    experiment = model_config.get("experiment")
    if experiment is None:
        # Older logs did not store an experiment label.
        experiment = "optimizer_grid" if optimizer_name else "architecture_grid"

    rows.append({
        "run": run,
        "experiment": experiment,
        "optimizer": optimizer_name,
        "units1": model_config.get("units1"),
        "units2": model_config.get("units2"),
        "units3": model_config.get("units3"),
        "epochs": training_config.get("epochs"),
        "learning_rate": training_config.get("optimizer_kwargs", {}).get("lr"),
        "weight_decay": training_config.get("optimizer_kwargs", {}).get("weight_decay"),
        "train_steps": training_config.get("train_steps"),
        "valid_steps": training_config.get("valid_steps"),
        "accuracy": get_last_scalar(run_path, "metric/Accuracy"),
        "train_loss": get_last_scalar(run_path, "Loss/train"),
        "test_loss": get_last_scalar(run_path, "Loss/test"),
    })

if not rows:
    raise RuntimeError(f"No completed experiment runs found in {logdir!r}")

results = pd.DataFrame(rows).sort_values("run").reset_index(drop=True)
results

In [ ]:
results["overfit_gap"] = results["test_loss"] - results["train_loss"]
nonzero_train_loss = results["train_loss"].where(results["train_loss"].ne(0))
results["overfit_ratio"] = results["test_loss"] / nonzero_train_loss
results["model_size"] = results[["units1", "units2", "units3"]].sum(
    axis=1,
    min_count=1,
)

completed_results = results.dropna(
    subset=["accuracy", "train_loss", "test_loss"]
).copy()
architecture_results = completed_results[
    completed_results["experiment"] == "architecture_grid"
].copy()
optimizer_results = completed_results[
    completed_results["experiment"] == "optimizer_grid"
].copy()

print(f"Architecture runs: {len(architecture_results)}")
print(f"Optimizer runs: {len(optimizer_results)}")

In [ ]:
filtered = completed_results[
    completed_results["overfit_ratio"] < 1.5
].sort_values("accuracy", ascending=False)

filtered[[
    "run", "experiment", "optimizer", "units1", "units2", "units3",
    "epochs", "learning_rate", "accuracy", "overfit_ratio",
]].head(10)

## Visualisations

Each plot uses a controlled subset. Parameters not shown in a comparison are held constant where possible, and duplicate runs are averaged rather than selecting only the best result.

In [ ]:
# Compare architectures with epochs and learning rate held constant.
architecture_comparison = architecture_results[
    (architecture_results["epochs"] == 10)
    & (architecture_results["learning_rate"] == 1e-3)
]

architecture_heatmap = architecture_comparison.pivot_table(
    index="units1",
    columns="units2",
    values="accuracy",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(architecture_heatmap, cmap="viridis")
for row in range(len(architecture_heatmap.index)):
    for column in range(len(architecture_heatmap.columns)):
        value = architecture_heatmap.iloc[row, column]
        if pd.notna(value):
            ax.text(column, row, f"{value:.3f}", ha="center", va="center", color="white")

ax.set_xticks(range(len(architecture_heatmap.columns)), architecture_heatmap.columns)
ax.set_yticks(range(len(architecture_heatmap.index)), architecture_heatmap.index)
ax.set_xlabel("units2")
ax.set_ylabel("units1")
ax.set_title("Mean accuracy by architecture (10 epochs, lr=0.001)")
fig.colorbar(image, ax=ax, label="accuracy")
plt.tight_layout()
plt.show()

In [ ]:
# Hold the architecture constant when comparing epochs and learning rates.
architecture_counts = architecture_results.groupby(["units1", "units2"]).size()
reference_units1, reference_units2 = architecture_counts.idxmax()
reference_architecture = architecture_results[
    (architecture_results["units1"] == reference_units1)
    & (architecture_results["units2"] == reference_units2)
]

epoch_lr_heatmap = reference_architecture.pivot_table(
    index="epochs",
    columns="learning_rate",
    values="accuracy",
    aggfunc="mean",
)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(epoch_lr_heatmap, cmap="viridis")
for row in range(len(epoch_lr_heatmap.index)):
    for column in range(len(epoch_lr_heatmap.columns)):
        value = epoch_lr_heatmap.iloc[row, column]
        if pd.notna(value):
            ax.text(column, row, f"{value:.3f}", ha="center", va="center", color="white")

ax.set_xticks(range(len(epoch_lr_heatmap.columns)), epoch_lr_heatmap.columns)
ax.set_yticks(range(len(epoch_lr_heatmap.index)), epoch_lr_heatmap.index)
ax.set_xlabel("learning rate")
ax.set_ylabel("epochs")
ax.set_title(f"Mean accuracy for units1={reference_units1}, units2={reference_units2}")
fig.colorbar(image, ax=ax, label="accuracy")
plt.tight_layout()
plt.show()

In [ ]:
optimizer_heatmap = optimizer_results.pivot_table(
    index="optimizer",
    columns="learning_rate",
    values="accuracy",
    aggfunc="mean",
).sort_index(axis=1)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(optimizer_heatmap, cmap="viridis")
for row in range(len(optimizer_heatmap.index)):
    for column in range(len(optimizer_heatmap.columns)):
        value = optimizer_heatmap.iloc[row, column]
        if pd.notna(value):
            ax.text(column, row, f"{value:.3f}", ha="center", va="center", color="white")

ax.set_xticks(range(len(optimizer_heatmap.columns)), optimizer_heatmap.columns)
ax.set_yticks(range(len(optimizer_heatmap.index)), optimizer_heatmap.index)
ax.set_xlabel("learning rate")
ax.set_ylabel("optimizer")
ax.set_title("Mean accuracy by optimizer and learning rate")
fig.colorbar(image, ax=ax, label="accuracy")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for experiment, group in completed_results.groupby("experiment"):
    ax.scatter(group["train_loss"], group["test_loss"], label=experiment, alpha=0.7)

loss_min = min(completed_results["train_loss"].min(), completed_results["test_loss"].min())
loss_max = max(completed_results["train_loss"].max(), completed_results["test_loss"].max())
ax.plot([loss_min, loss_max], [loss_min, loss_max], linestyle="--", color="black")
ax.set_xlabel("train loss")
ax.set_ylabel("test loss")
ax.set_title("Train loss versus test loss")
ax.legend()
plt.tight_layout()
plt.show()
## Experiment: Dropout and Normalization

**Hypotheses**:\n- Dropout reduces overfitting; moderate dropout (0.2) improves validation accuracy for larger networks.\n- BatchNorm/LayerNorm stabilises training and reduces sensitivity to higher learning rates; BatchNorm may perform better with larger batches.

We run a grid over `dropout_rate` and `normalization` while keeping other training settings identical. Results are logged by the existing `Trainer`.

In [ ]:
# Grid: dropout x normalization x architecture (sanity-size grid)
dropout_rates = [0.0, 0.2, 0.5]
normalizations = ["none", "batch", "layer"]
architectures = [(256, 128, 64), (128, 64, 32)]

grid_settings = TrainerSettings(
    epochs=5,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

for units in architectures:
    for dr in dropout_rates:
        for norm in normalizations:
            u1, u2, u3 = units
            model = NeuralNetwork(
                num_classes=10,
                units1=u1,
                units2=u2,
                units3=u3,
                dropout_rate=dr,
                normalization=norm,
            )
            model.experiment = "dropout_norm_grid"
            model.optimizer = "Adam"
            trainer = Trainer(
                model=model,
                settings=grid_settings,
                loss_fn=loss_fn,
                optimizer=optim.Adam,
                traindataloader=trainstreamer,
                validdataloader=validstreamer,
                scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            )
            trainer.loop()

### Analyse-aanpak

De bestaande log-lezer in deze notebook leest `model.toml` en `settings.toml`. Omdat `dropout_rate` en `normalization` publieke attributen zijn, verschijnen ze automatisch in de `results` DataFrame en kunnen we meteen pivot/tabellen maken om hun effect op `accuracy`, `train_loss` en `test_loss` te onderzoeken.

Tip: begin met één sanity-run (bijv. `(128,64,32)`, dropout=0.2, normalization='batch') om te controleren of alles goed logt voordat je de hele grid uitvoert.

## Visualisaties: Dropout vs Normalization

De onderstaande cellen visualiseren het effect van `dropout_rate` en `normalization` op `accuracy` en `overfit_gap` per architectuur. Gebruik TensorBoard voor gedetailleerde trainingcurves.

In [ ]:
# Pivot heatmaps: accuracy for dropout x normalization per architecture
df = completed_results[completed_results['experiment'] == 'dropout_norm_grid'].copy()
if df.empty:
    print('No dropout_norm_grid runs found in modellogs')
else:
    for (u1, u2, u3), group in df.groupby(['units1','units2','units3']):
        pivot = group.pivot_table(index='dropout_rate', columns='normalization', values='accuracy', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(6, 4))
        image = ax.imshow(pivot, cmap='viridis', aspect='auto')
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f'{val:.3f}', ha='center', va='center', color='white')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_xlabel('normalization')
        ax.set_ylabel('dropout_rate')
        ax.set_title(f'Accuracy for units=({u1},{u2},{u3})')
        fig.colorbar(image, ax=ax, label='accuracy')
        plt.tight_layout()
        plt.show()

In [ ]:
# Overfit gap by dropout and normalization (aggregated)
df = completed_results[completed_results['experiment'] == 'dropout_norm_grid'].copy()
if df.empty:
    print('No dropout_norm_grid runs found in modellogs')
else:
    agg = df.pivot_table(index='dropout_rate', columns='normalization', values='overfit_gap', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(6, 4))
    image = ax.imshow(agg, cmap='coolwarm', aspect='auto')
    for i in range(agg.shape[0]):
        for j in range(agg.shape[1]):
            val = agg.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, f'{val:.3f}', ha='center', va='center', color='black')
    ax.set_xticks(range(len(agg.columns)))
    ax.set_xticklabels(agg.columns)
    ax.set_yticks(range(len(agg.index)))
    ax.set_yticklabels(agg.index)
    ax.set_xlabel('normalization')
    ax.set_ylabel('dropout_rate')
    ax.set_title('Mean overfit_gap by dropout and normalization')
    fig.colorbar(image, ax=ax, label='overfit_gap')
    plt.tight_layout()
    plt.show()

# Optional: boxplot of accuracy by normalization for each dropout rate
    for dr in sorted(df['dropout_rate'].unique()):
        sub = df[df['dropout_rate'] == dr]
        fig, ax = plt.subplots(figsize=(6,4))
        sub.boxplot(column='accuracy', by='normalization', ax=ax)
        ax.set_title(f'Accuracy distribution at dropout={dr}')
        ax.set_xlabel('normalization')
        ax.set_ylabel('accuracy')
        plt.suptitle('')
        plt.tight_layout()
        plt.show()

**In-Notebook Figures**

De gegenereerde figuren staan hieronder; open de notebook om ze inline te bekijken.

- ![Accuracy units 128_64_32](figures/accuracy_units_128_64_32.png)
- ![Accuracy units 256_128_64](figures/accuracy_units_256_128_64.png)
- ![Overfit gap](figures/overfit_gap.png)
- ![Accuracy box dropout 0.0](figures/accuracy_box_dropout_0.0.png)
- ![Accuracy box dropout 0.2](figures/accuracy_box_dropout_0.2.png)
- ![Accuracy box dropout 0.5](figures/accuracy_box_dropout_0.5.png)

Je kunt de output later verwijderen (strippen) met tools zoals `nbstripout` of via de notebook-UI als je de notebook zonder ingesloten outputs wilt delen.